# Домашнее задание 7.1 — остывание стержня

**Жёлтый уровень — схема Кранка—Николсона и разрывный начальный профиль**

Рассматривается одномерное уравнение теплопроводности

$$
u_t=u_{xx},\qquad 0\le x\le 1,
$$

с условиями Дирихле $u(0,t)=u(1,t)=0$. В начальный момент нагрет только центральный участок стержня.

$$
u(x,0)=
\begin{cases}
1, & 0.4\le x\le 0.6,\\
0, & \text{вне этого участка}.
\end{cases}
$$

**Цель.** Сравнить неявный Эйлер и схему Кранка—Николсона при $r=\Delta t/\Delta x^2=0.4,\ 2,\ 10$ на одной пространственной сетке. FTCS используется при $r=0.4$ и для расчёта на малом временном шаге.

## 1. Разностные схемы

Пусть $x_j=j\Delta x$, $j=0,\ldots,N$, а $t_n=n\Delta t$. Для внутренних узлов обозначим центральную разность без деления на $\Delta x^2$ через $D$.

$$
(Du^n)_j=u_{j-1}^n-2u_j^n+u_{j+1}^n.
$$

**Явная схема FTCS**

$$
u^{n+1}=u^n+rDu^n,\qquad r\le\tfrac12.
$$

**Неявный Эйлер**

$$
(I-rD)u^{n+1}=u^n.
$$

**Схема Кранка—Николсона**

$$
\left(I-\frac r2D\right)u^{n+1}
=\left(I+\frac r2D\right)u^n.
$$

Схема Кранка—Николсона усредняет пространственную разность на старом и новом временных слоях. Матрица в левой части трёхдиагональная с диагональю $1+r$ и соседними диагоналями $-r/2$. Обе неявные схемы решаются методом прогонки. Граничные узлы на каждом слое равны нулю.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Параметры эксперимента
N = 100                                # число пространственных интервалов
R_VALUES = [0.4, 2, 10]
R_REF = 0.01                           # малый шаг для расчёта сравнения
T_VALUES = [0.001, 0.002, 0.003, 0.01]  # общие моменты сравнения

plt.rcParams["figure.figsize"] = (8, 4.8)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

In [ ]:
def thomas_algorithm(lower, diagonal, upper, rhs):
    """Решение трёхдиагональной СЛАУ методом прогонки.

    lower и upper имеют длину n-1, diagonal и rhs — длину n.
    Входные массивы не изменяются.
    """
    n = len(diagonal)
    if n == 0:
        return np.array([], dtype=float)
    if len(rhs) != n or len(lower) != n - 1 or len(upper) != n - 1:
        raise ValueError("Несогласованные размеры диагоналей трёхдиагональной матрицы")

    c_star = np.zeros(max(n - 1, 0), dtype=float)
    d_star = np.zeros(n, dtype=float)

    denom = diagonal[0]
    if abs(denom) < 1e-14:
        raise ZeroDivisionError("Нулевой ведущий элемент в методе прогонки")
    if n > 1:
        c_star[0] = upper[0] / denom
    d_star[0] = rhs[0] / denom

    for i in range(1, n):
        denom = diagonal[i] - lower[i - 1] * c_star[i - 1]
        if abs(denom) < 1e-14:
            raise ZeroDivisionError("Нулевой ведущий элемент в методе прогонки")
        if i < n - 1:
            c_star[i] = upper[i] / denom
        d_star[i] = (rhs[i] - lower[i - 1] * d_star[i - 1]) / denom

    solution = np.empty(n, dtype=float)
    solution[-1] = d_star[-1]
    for i in range(n - 2, -1, -1):
        solution[i] = d_star[i] - c_star[i] * solution[i + 1]
    return solution


def ftcs_step(u, r):
    """Один шаг явной схемы FTCS при u(0,t)=u(1,t)=0."""
    if r > 0.5 + 1e-12:
        raise ValueError(f"FTCS неустойчива: r={r:.4g} > 0.5")
    u_new = np.zeros_like(u)
    u_new[1:-1] = u[1:-1] + r * (u[:-2] - 2.0 * u[1:-1] + u[2:])
    return u_new


def backward_euler_step(u, r):
    """Один шаг неявного Эйлера; СЛАУ решается методом прогонки."""
    n_inner = len(u) - 2
    diagonal = np.full(n_inner, 1.0 + 2.0 * r)
    lower = np.full(n_inner - 1, -r)
    upper = np.full(n_inner - 1, -r)
    u_new = np.zeros_like(u)
    u_new[1:-1] = thomas_algorithm(lower, diagonal, upper, u[1:-1])
    return u_new

In [ ]:
def crank_nicolson_step(u, r):
    """Один шаг схемы Кранка—Николсона; СЛАУ решается методом прогонки."""
    n_inner = len(u) - 2
    diagonal = np.full(n_inner, 1.0 + r)
    lower = np.full(n_inner - 1, -0.5 * r)
    upper = np.full(n_inner - 1, -0.5 * r)
    rhs = u[1:-1] + 0.5 * r * (u[:-2] - 2.0 * u[1:-1] + u[2:])
    u_new = np.zeros_like(u)
    u_new[1:-1] = thomas_algorithm(lower, diagonal, upper, rhs)
    return u_new


def solve_history(u_initial, dx, target_time, step, r_nominal):
    """Сохранить временные слои, при необходимости укоротив последний шаг."""
    dt = r_nominal * dx**2
    n_full = int(np.floor(target_time / dt + 1e-12))
    times = np.arange(n_full + 1) * dt
    if np.isclose(times[-1], target_time, rtol=0, atol=1e-14):
        times[-1] = target_time
    else:
        times = np.append(times, target_time)

    profiles = np.empty((len(times), len(u_initial)))
    profiles[0] = u_initial
    for n, dt_step in enumerate(np.diff(times)):
        profiles[n + 1] = step(profiles[n], dt_step / dx**2)
    return times, profiles


def profile_at_time(history, target_time, step, dx):
    """Взять готовый слой или сделать укороченный шаг до нужного момента."""
    times, profiles = history
    n = np.searchsorted(times, target_time + 1e-14, side="right") - 1
    dt_step = target_time - times[n]
    if abs(dt_step) <= 1e-14:
        return profiles[n]
    return step(profiles[n], dt_step / dx**2)

## 2. Расчёт на одной сетке

Берём $N=100$ интервалов, поэтому $\Delta x=0.01$, а края нагретого участка совпадают с узлами сетки. Оба крайних узла участка входят в начальный профиль с температурой 1. Номинальные шаги $\Delta t=r\Delta x^2$ равны $4\cdot10^{-5}$, $2\cdot10^{-4}$ и $10^{-3}$.

Для сравнения используем FTCS на той же сетке при $r_{\mathrm{мал}}=0.01$ ($\Delta t=10^{-6}$). Так можно исследовать влияние шага по времени при фиксированной пространственной аппроксимации. Во всех сравнениях берём профили в один и тот же физический момент. При необходимости укорачиваем последний шаг до этого момента.

In [ ]:
dx = 1.0 / N
x = np.linspace(0.0, 1.0, N + 1)
nodes = np.arange(N + 1)
u_initial = ((5 * nodes >= 2 * N) & (5 * nodes <= 3 * N)).astype(float)

methods = {
    "FTCS": ftcs_step,
    "Неявный Эйлер": backward_euler_step,
    "Кранк—Николсон": crank_nicolson_step,
}

histories = {}
for r in R_VALUES:
    for method, step in methods.items():
        if method == "FTCS" and r > 0.5:
            continue
        histories[(r, method)] = solve_history(u_initial, dx, T_VALUES[-1], step, r)

reference = solve_history(u_initial, dx, T_VALUES[-1], ftcs_step, R_REF)
reference_fine = solve_history(u_initial, dx, T_VALUES[-1], ftcs_step, R_REF / 2)

print("Расчёты завершены.")
print(f"Количество комбинаций (r, метод): {len(histories)}")
print(f"Шаг расчёта сравнения: {R_REF * dx**2:.1e}")

## 3. Профили первых временных слоёв

Столбцы соответствуют первым трём шагам при данном $r$. В верхней строке показан весь стержень, в следующих двух — увеличенные фрагменты около $x=0.4$ и $x=0.6$. В каждом столбце чёрная линия показывает расчёт на малом шаге в момент, указанный над графиком.

In [ ]:
styles = {
    "FTCS": dict(color="C0", linestyle="--", marker="o", markersize=3),
    "Неявный Эйлер": dict(color="C1", linestyle="-.", marker="s", markersize=3),
    "Кранк—Николсон": dict(color="C2", linestyle=":", marker="^", markersize=3.6),
}
views = [(0, 1), (0.33, 0.47), (0.53, 0.67)]
row_labels = ["Весь стержень", "Около x = 0.4", "Около x = 0.6"]

for r in R_VALUES:
    fig, axes = plt.subplots(3, 3, figsize=(13, 8.5), sharey=True)
    fig.suptitle(f"Первые временные слои, r = {r:g}, N = {N}", fontsize=14)

    for column, n in enumerate([1, 2, 3]):
        t = histories[(r, "Неявный Эйлер")][0][n]
        u_ref = profile_at_time(reference, t, ftcs_step, dx)

        for row, (left, right) in enumerate(views):
            ax = axes[row, column]
            ax.plot(x, u_ref, color="black", linewidth=1.8, label="Малый шаг")
            for method in methods:
                if (r, method) in histories:
                    u = histories[(r, method)][1][n]
                    ax.plot(x, u, linewidth=1.2, label=method, **styles[method])

            ax.set_xlim(left, right)
            ax.set_xlabel("x")
            if row:
                edge = 0.4 if row == 1 else 0.6
                ax.axvline(edge, color="0.6", linestyle="--", linewidth=0.8)
            if column == 0:
                ax.set_ylabel(f"{row_labels[row]}\nu(x,t)")
            if row == 0:
                ax.set_title(f"Слой {n}, t = {t:g}")

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(labels), fontsize=9)
    fig.tight_layout(rect=(0, 0.055, 1, 0.96))
    plt.show()

## 4. Минимум и максимум решения

На каждом временном слое берём минимум и максимум по всем узлам, включая концы стержня. Точки кривых соответствуют фактическим временным слоям каждой схемы. Отдельные маркеры отмечают слои для $r=10$. Значения температур используются без обрезки.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
times_ref, profiles_ref = reference
colors = {0.4: "C0", 2: "C1", 10: "C2"}

for ax, reducer, title in zip(axes, [np.min, np.max], ["Минимум", "Максимум"]):
    for (r, method), (times, profiles) in histories.items():
        marker = styles[method]["marker"] if r == 10 else None
        ax.plot(times, reducer(profiles, axis=1), color=colors[r],
                linestyle=styles[method]["linestyle"], linewidth=1.5,
                marker=marker, markersize=3, label=f"{method}, r = {r:g}")
    ax.plot(times_ref, reducer(profiles_ref, axis=1), color="black",
            linewidth=1.5, label="Малый шаг")
    ax.set_title(title)
    ax.set_xlabel("t")
    ax.set_xlim(0, T_VALUES[-1])
    ax.ticklabel_format(axis="x", style="sci", scilimits=(-3, -3))

axes[0].set_ylabel(r"$\min_j u_j(t)$")
axes[1].set_ylabel(r"$\max_j u_j(t)$")
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=8)
fig.tight_layout(rect=(0, 0.18, 1, 1))
plt.show()

## 5. Сравнение с расчётом на малом шаге

Для неявных схем сравниваем профили при всех трёх $r$ в общие моменты $t=0.001,\ 0.002,\ 0.003,\ 0.01$. В таблице приведено максимальное расхождение на пространственной сетке.

$$
E_\infty(r,t)=\max_j \left|u_j(r,t)-u_{j,\mathrm{мал}}(t)\right|.
$$

Для проверки расчёта сравнения дополнительно уменьшаем его шаг вдвое. Расхождение между двумя расчётами FTCS на малых шагах показывает чувствительность опорного профиля к дальнейшему уменьшению $\Delta t$.

In [ ]:
error_rows = []
for t in T_VALUES:
    u_ref = profile_at_time(reference, t, ftcs_step, dx)
    for r in R_VALUES:
        row = {"t": t, "r": r}
        for method in ["Неявный Эйлер", "Кранк—Николсон"]:
            u = profile_at_time(histories[(r, method)], t, methods[method], dx)
            row[method] = np.max(np.abs(u - u_ref))
        error_rows.append(row)

errors = pd.DataFrame(error_rows)
display(errors.style.format({"t": "{:.3f}", "r": "{:g}",
                             "Неявный Эйлер": "{:.6e}", "Кранк—Николсон": "{:.6e}"}))

# Проверка на всех показанных моментах, включая первые три слоя для каждого r
check_times = sorted(set(T_VALUES) | {
    t for r in R_VALUES for t in histories[(r, "Неявный Эйлер")][0][1:4]
})
reference_difference = max(
    np.max(np.abs(profile_at_time(reference, t, ftcs_step, dx)
                  - profile_at_time(reference_fine, t, ftcs_step, dx)))
    for t in check_times
)
print(f"Наибольшее расхождение при уменьшении малого шага вдвое: {reference_difference:.3e}")

Вблизи левого края нагретого участка профиль на малом шаге возрастает от узла $x=0.39$ к узлу $x=0.40$. Разность $u(0.40,t)-u(0.39,t)$ помогает увидеть, как схема Кранка—Николсона при $r=10$ меняет направление перепада в первых слоях.

In [ ]:
edge_index = 2 * N // 5
edge_rows = []
for t in T_VALUES[:3]:
    row = {"t": t}
    for label, history, step in [
        ("Малый шаг", reference, ftcs_step),
        ("Неявный Эйлер, r = 10", histories[(10, "Неявный Эйлер")], backward_euler_step),
        ("Кранк—Николсон, r = 10", histories[(10, "Кранк—Николсон")], crank_nicolson_step),
    ]:
        u = profile_at_time(history, t, step, dx)
        row[label] = u[edge_index] - u[edge_index - 1]
    edge_rows.append(row)

display(pd.DataFrame(edge_rows).style.format("{:.6f}"))

## 6. Ответы на вопросы и выводы

**Гарантирует ли безусловная устойчивость точность и отсутствие осцилляций?**

Безусловная устойчивость ограничивает рост возмущений при любом шаге, но ни точность, ни монотонность профиля из этого не следуют. Для пространственной гармоники с собственным значением $-\lambda$ дискретного оператора $D/\Delta x^2$ положим $z=\lambda\Delta t\ge 0$. Множители амплитуды за один шаг равны

$$
g_{\mathrm{Э}}(z)=\frac{1}{1+z},
\qquad
g_{\mathrm{КН}}(z)=\frac{1-z/2}{1+z/2}.
$$

Оба множителя по модулю не превосходят единицы. При больших $z$ неявный Эйлер сильно гасит гармонику, поскольку $g_{\mathrm{Э}}\to 0$. У схемы Кранка—Николсона при $z>2$ гармоника меняет знак на каждом шаге, а при больших $z$ её модуль убывает медленно, поскольку $g_{\mathrm{КН}}\to -1$. Разрывы начального профиля возбуждают коротковолновые гармоники, поэтому этот эффект особенно заметен около краёв нагретого участка.

**Что видно в расчётах?**

- При $r=10$ на первых трёх слоях Кранка—Николсона разность $u(0.40,t)-u(0.39,t)$ меняет знак ($-0.564$, $+0.584$, $-0.387$). В расчёте на малом шаге эта разность остаётся положительной. На увеличенных фрагментах видны локальные провалы и пики у обоих краёв участка.
- При $t=0.001$ и $r=10$ максимальное расхождение с расчётом на малом шаге составляет $0.327$ для Кранка—Николсона и $0.0622$ для неявного Эйлера. Уменьшение $r$ до $0.4$ приближает оба профиля к расчёту на малом шаге. Самые малые расхождения в таблице зависят от точности опорного расчёта.
- На всех показанных слоях минимум равен нулю, а максимум не превышает единицы. При этом профиль Кранка—Николсона при $r=10$ осциллирует внутри допустимого диапазона температур. Глобальные минимум и максимум сами по себе не выявляют такие осцилляции.